# 05 — Retrieval

This notebook demonstrates the NeuroForge retrieval system, which combines multiple
strategies for finding relevant knowledge:

1. **Semantic Search** — embed a query and find the top-k similar chunks
2. **Filtered Search** — semantic search constrained by metadata (topic, difficulty)
3. **Graph-Based Retrieval** — traverse the knowledge graph to find concept + prerequisites + related
4. **Hybrid Retrieval** — combine semantic + graph approaches, deduplicate, and rank

In [ ]:
import sys
sys.path.insert(0, "..")

import chromadb
from models import Chunk, ChunkMetadata, Concept, ConceptRelationship, Difficulty
from src.store import VectorStore, KnowledgeGraph
from src.retrieval import Retriever

## Setup: Create Sample Data

We'll use an ephemeral ChromaDB client so no files are written to disk.

In [ ]:
# Create ephemeral stores
client = chromadb.Client()
vector_store = VectorStore(client=client)
vector_store.init_collections()
knowledge_graph = KnowledgeGraph()

print("Stores initialized.")

In [ ]:
# Sample chunks simulating a machine learning textbook
chunks = [
    Chunk(
        id="chunk_001",
        content="Machine learning is a subset of artificial intelligence that enables systems to learn patterns from data without being explicitly programmed.",
        document_id="doc_ml",
        chunk_index=0,
        metadata=ChunkMetadata(section_heading="Introduction to ML", page_number=1, token_count=25, start_char=0, end_char=140),
    ),
    Chunk(
        id="chunk_002",
        content="Neural networks are computing systems inspired by biological neural networks. They consist of layers of interconnected nodes called neurons.",
        document_id="doc_ml",
        chunk_index=1,
        metadata=ChunkMetadata(section_heading="Neural Networks", page_number=5, token_count=24, start_char=141, end_char=280),
    ),
    Chunk(
        id="chunk_003",
        content="Gradient descent is an iterative optimization algorithm used to find the minimum of a function. It is fundamental to training machine learning models.",
        document_id="doc_ml",
        chunk_index=2,
        metadata=ChunkMetadata(section_heading="Optimization", page_number=12, token_count=28, start_char=281, end_char=435),
    ),
    Chunk(
        id="chunk_004",
        content="Backpropagation is the algorithm for computing gradients of the loss function with respect to each weight, using the chain rule of calculus.",
        document_id="doc_ml",
        chunk_index=3,
        metadata=ChunkMetadata(section_heading="Backpropagation", page_number=15, token_count=26, start_char=436, end_char=575),
    ),
    Chunk(
        id="chunk_005",
        content="Supervised learning trains models using labeled datasets where both inputs and expected outputs are provided during training.",
        document_id="doc_ml",
        chunk_index=4,
        metadata=ChunkMetadata(section_heading="Learning Paradigms", page_number=20, token_count=20, start_char=576, end_char=700),
    ),
    Chunk(
        id="chunk_006",
        content="Convolutional neural networks (CNNs) are specialized for processing grid-like data such as images, using convolutional layers for feature extraction.",
        document_id="doc_ml",
        chunk_index=5,
        metadata=ChunkMetadata(section_heading="CNN Architecture", page_number=25, token_count=26, start_char=701, end_char=855),
    ),
    Chunk(
        id="chunk_007",
        content="Regularization techniques like dropout and L2 regularization prevent overfitting by adding constraints or noise during training.",
        document_id="doc_ml",
        chunk_index=6,
        metadata=ChunkMetadata(section_heading="Regularization", page_number=30, token_count=20, start_char=856, end_char=985),
    ),
]

vector_store.add_chunks(chunks)
print(f"Added {len(chunks)} chunks to vector store.")

In [ ]:
# Sample concepts
concepts = [
    Concept(
        id="concept_ml",
        name="Machine Learning",
        definition="A subset of AI enabling systems to learn from data.",
        topics=["artificial_intelligence", "data_science"],
        difficulty=Difficulty.EASY,
        prerequisites=[],
        keywords=["ML", "learning", "AI", "data"],
        source_chunk_ids=["chunk_001"],
    ),
    Concept(
        id="concept_nn",
        name="Neural Networks",
        definition="Computing systems inspired by biological neural networks.",
        topics=["deep_learning", "artificial_intelligence"],
        difficulty=Difficulty.MEDIUM,
        prerequisites=["concept_ml"],
        keywords=["neurons", "layers", "deep learning"],
        source_chunk_ids=["chunk_002"],
    ),
    Concept(
        id="concept_gd",
        name="Gradient Descent",
        definition="Optimization algorithm to minimize a loss function iteratively.",
        topics=["optimization", "machine_learning"],
        difficulty=Difficulty.MEDIUM,
        prerequisites=["concept_ml"],
        keywords=["gradient", "optimization", "loss", "learning rate"],
        source_chunk_ids=["chunk_003"],
    ),
    Concept(
        id="concept_bp",
        name="Backpropagation",
        definition="Algorithm for computing gradients using the chain rule.",
        topics=["deep_learning", "optimization"],
        difficulty=Difficulty.HARD,
        prerequisites=["concept_nn", "concept_gd"],
        keywords=["backprop", "chain rule", "gradients", "weights"],
        source_chunk_ids=["chunk_004"],
    ),
    Concept(
        id="concept_cnn",
        name="Convolutional Neural Networks",
        definition="Specialized neural networks for grid-like data such as images.",
        topics=["deep_learning", "computer_vision"],
        difficulty=Difficulty.HARD,
        prerequisites=["concept_nn"],
        keywords=["CNN", "convolution", "filters", "feature maps"],
        source_chunk_ids=["chunk_006"],
    ),
]

vector_store.add_concepts(concepts)
knowledge_graph.add_concepts(concepts)
print(f"Added {len(concepts)} concepts.")

In [ ]:
# Relationships
relationships = [
    ConceptRelationship(source_concept="concept_ml", target_concept="concept_nn", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="concept_ml", target_concept="concept_gd", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="concept_nn", target_concept="concept_bp", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="concept_gd", target_concept="concept_bp", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="concept_nn", target_concept="concept_cnn", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="concept_nn", target_concept="concept_gd", relationship_type="related"),
]

knowledge_graph.add_relationships(relationships)
print(f"Added {len(relationships)} relationships.")
print(f"Graph has {len(knowledge_graph)} nodes and {knowledge_graph.graph.number_of_edges()} edges.")

In [ ]:
# Create the Retriever
retriever = Retriever(vector_store=vector_store, knowledge_graph=knowledge_graph)
print("Retriever ready.")

## 1. Semantic Search

Pure vector similarity — embed the query and find the closest chunks.

In [ ]:
query = "How do neural networks learn?"
results = retriever.semantic_search(query, top_k=3)

print(f"Query: '{query}'\n")
print(f"Top {len(results)} results:\n")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['id']}] (score: {r['score']:.4f})")
    print(f"     {r['content'][:100]}...")
    print(f"     Section: {r['metadata'].get('section_heading', 'N/A')}")
    print()

In [ ]:
# Another semantic search example
query2 = "optimization and loss minimization"
results2 = retriever.semantic_search(query2, top_k=3)

print(f"Query: '{query2}'\n")
for i, r in enumerate(results2, 1):
    print(f"  {i}. [{r['id']}] (score: {r['score']:.4f})")
    print(f"     {r['content'][:100]}")
    print()

## 2. Filtered Search (Metadata Constraints)

Search concepts filtered by topic and/or difficulty, then retrieve associated chunks.

In [ ]:
# Filter by difficulty: easy concepts only
results_easy = retriever.filtered_search("learning basics", difficulty="easy")

print("Filtered by difficulty='easy':\n")
for r in results_easy:
    print(f"  [{r['id']}] score={r['score']:.4f}")
    print(f"  {r['content'][:80]}...")
    print()

In [ ]:
# Filter by topic: deep_learning
results_dl = retriever.filtered_search("architectures", topic="deep_learning")

print("Filtered by topic='deep_learning':\n")
for r in results_dl:
    print(f"  [{r['id']}] score={r['score']:.4f}")
    print(f"  {r['content'][:80]}...")
    print()

In [ ]:
# Combined filter: topic + difficulty
results_combined = retriever.filtered_search(
    "advanced techniques", topic="optimization", difficulty="hard"
)

print("Filtered by topic='optimization' AND difficulty='hard':\n")
if results_combined:
    for r in results_combined:
        print(f"  [{r['id']}] score={r['score']:.4f}")
        print(f"  {r['content'][:80]}...")
else:
    print("  No results (expected for this narrow filter).")

## 3. Graph-Based Retrieval

Start from a concept, expand via prerequisites and related concepts, fetch chunks.

In [ ]:
# Retrieve everything related to Backpropagation
concept_id = "concept_bp"
graph_results = retriever.graph_retrieval(concept_id)

print(f"Graph retrieval for '{concept_id}' (Backpropagation):\n")
print(f"Prerequisites: {knowledge_graph.get_prerequisites(concept_id)}")
print(f"Related: {knowledge_graph.get_related(concept_id)}")
print(f"\nRetrieved {len(graph_results)} chunks:\n")
for r in graph_results:
    print(f"  [{r['id']}] score={r['score']:.2f}")
    print(f"  {r['content'][:80]}...")
    print()

In [ ]:
# Graph retrieval for CNN (depends on neural networks → machine learning)
concept_id2 = "concept_cnn"
graph_results2 = retriever.graph_retrieval(concept_id2)

print(f"Graph retrieval for '{concept_id2}' (CNN):\n")
print(f"Prerequisites: {knowledge_graph.get_prerequisites(concept_id2)}")
print(f"\nRetrieved {len(graph_results2)} chunks:\n")
for r in graph_results2:
    print(f"  [{r['id']}] score={r['score']:.2f} — {r['content'][:60]}...")

## 4. Hybrid Retrieval

Combines semantic search with graph expansion. Steps:
1. Run semantic search for direct matches
2. Identify related concepts via the concepts collection
3. Expand via graph (prerequisites + related)
4. Merge, deduplicate by chunk ID, sort by score

In [ ]:
query_hybrid = "How does backpropagation work in neural networks?"
hybrid_results = retriever.hybrid_retrieval(query_hybrid, top_k=5)

print(f"Hybrid retrieval: '{query_hybrid}'\n")
print(f"Returned {len(hybrid_results)} results (deduplicated, ranked):\n")
for i, r in enumerate(hybrid_results, 1):
    print(f"  {i}. [{r['id']}] score={r['score']:.4f}")
    print(f"     {r['content'][:90]}...")
    print()

In [ ]:
# Compare semantic-only vs hybrid for the same query
sem_only = retriever.semantic_search(query_hybrid, top_k=5)

print("=== Semantic Only ===")
for r in sem_only:
    print(f"  [{r['id']}] score={r['score']:.4f}")

print("\n=== Hybrid ===")
for r in hybrid_results:
    print(f"  [{r['id']}] score={r['score']:.4f}")

# Hybrid should surface additional context from graph expansion
hybrid_ids = set(r['id'] for r in hybrid_results)
sem_ids = set(r['id'] for r in sem_only)
extra = hybrid_ids - sem_ids
if extra:
    print(f"\nHybrid found {len(extra)} additional chunks via graph: {extra}")
else:
    print("\nBoth returned the same chunks (graph didn't add new ones for this query).")

## Summary

| Strategy | Best For |
|----------|----------|
| **Semantic** | Quick similarity lookups, general questions |
| **Filtered** | Narrowing results by topic or difficulty level |
| **Graph** | Understanding a concept in context (with prerequisites) |
| **Hybrid** | Comprehensive retrieval combining breadth + depth |

The `Retriever` class provides a unified interface to all strategies, making it
easy to switch or combine approaches depending on the user's learning context.